# 04 — Run it six ways

Copy the working functions from notebook 03 into the cells below. Yes, copy-paste. That's fine.

**The one discipline that matters:** before recording any numbers, hit *Restart kernel → Run
all* and let this run top to bottom. If the numbers only appear when cells are run in some
particular order, they aren't numbers.

The response cache makes that cheap — a second full run costs zero API calls.

In [1]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


working from: /content/pa-appeal


In [12]:
# from getpass import getpass
# import os

# if not os.environ.get("GEMINI_API_KEY"):
#     os.environ["GEMINI_API_KEY"] = getpass("Paste Gemini API key: ")

# assert os.environ.get("GEMINI_API_KEY"), "No API key found"
# print("Key loaded")

from getpass import getpass
import os

os.environ["GEMINI_API_KEY"] = getpass("Paste the NEW Gemini API key: ")

assert os.environ["GEMINI_API_KEY"], "No API key entered"
print("New API key loaded")


New API key loaded


In [3]:
policies = {p.stem: p.read_text() for p in sorted(Path("data/policies").glob("*.md"))}
criteria  = json.load(open("data/criteria.json"))
cases     = json.load(open("data/cases.json"))
Path("data/results").mkdir(parents=True, exist_ok=True)


# Working pipeline functions

In [13]:
# ---- from 03_pipeline: cell 4 ----
import hashlib, random, time
from typing import Literal

from pydantic import BaseModel, Field

CACHE = Path("data/cache"); CACHE.mkdir(parents=True, exist_ok=True)

# Stable, low-cost model suited to classification and structured extraction.
MODEL = "gemini-3.1-flash-lite"


class Decision(BaseModel):
    criterion_id: str
    label: Literal["met", "unmet", "insufficient_evidence"]
    evidence_quote: str = Field(
        description="Exact supporting quotation copied from the supplied policy text")
    source_doc_id: str
    reasoning: str
    model_reported_confidence: float = Field(
        ge=0.0, le=1.0,
        description=(
            "The model's own 0-1 confidence in this label. SELF-REPORTED AND "
            "UNCALIBRATED -- recorded for exploration only, never as a reliability "
            "estimate. The trustworthy signal in this project is quote_status, which "
            "is checked by string matching against the policy text."))


class DecisionBatch(BaseModel):
    decisions: list[Decision]


DECISION_SCHEMA = DecisionBatch.model_json_schema()


def ask(prompt, system="", model=MODEL, force=False, response_schema=None):
    """Ask the LLM, but only once per unique prompt. Repeats come off disk.

    This is the single most useful thing on a free tier. Restart & Run All costs
    zero API calls once the cache is warm.
    """
    schema_key = json.dumps(response_schema, sort_keys=True) if response_schema else ""
    key = hashlib.sha256(
        f"{model}|{system}|{schema_key}|{prompt}".encode()).hexdigest()[:16]
    f = CACHE / f"{key}.json"
    if f.exists() and not force:
        return json.loads(f.read_text())["response"]

    from google import genai   # SDK surface changes - check current docs if this errors
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    for attempt in range(5):
        try:
            config = {
                "system_instruction": system,
                "temperature": 0,
                "response_mime_type": "application/json",
            }
            if response_schema is not None:
                config["response_json_schema"] = response_schema

            r = client.models.generate_content(
                model=model,
                contents=prompt,
                config=config,
            )
            text = r.text
            break
        except Exception as e:
            message = str(e).lower()
            retryable = any(token in message for token in (
                "429", "503", "resource_exhausted", "unavailable"))
            if not retryable or attempt == 4:
                raise
            wait = 5 * (2 ** attempt) + random.random()
            print(f"temporary API error; retrying in {wait:.1f}s")
            time.sleep(wait)

    f.write_text(json.dumps({"model": model, "system": system,
                             "prompt": prompt, "response": text}, indent=2))
    return text

def ask_json(prompt, system="", **kw):
    raw = ask(
        prompt, system, response_schema=DECISION_SCHEMA, **kw)
    return DecisionBatch.model_validate_json(raw).model_dump()

# ---- from 03_pipeline: cell 6 ----
def chunk_fixed(text, doc_id, size=512, overlap=64):
    words, step, out = text.split(), size - overlap, []
    for i in range(0, max(1, len(words)), step):
        w = words[i:i + size]
        if not w:
            break
        out.append({"id": f"{doc_id}::fix::{len(out)}", "doc": doc_id, "text": " ".join(w),
                    "criterion": None})
    return out

def chunk_headings(text, doc_id, min_chars=200):
    marks = list(re.finditer(r"^#{1,6}\s+(.*)$", text, re.M))
    if not marks:
        return chunk_fixed(text, doc_id)
    out = []
    for n, m in enumerate(marks):
        end = marks[n + 1].start() if n + 1 < len(marks) else len(text)
        body = text[m.start():end].strip()
        if len(body) < min_chars and out:
            out[-1]["text"] += "\n\n" + body
            continue
        out.append({"id": f"{doc_id}::sec::{len(out)}", "doc": doc_id, "text": body,
                    "criterion": None})
    return out

def chunk_by_criteria(criteria, policies):
    # text_core, not text: the raw quotes carry markdown list markers ("1. ")
    # that the model never reproduces. Indexing the clean sentence keeps the
    # normalized quotation check honest while keeping each rule intact.
    out = [{"id": f"crit::{c['id']}", "doc": c["source"],
            "text": c.get("text_core") or c["text"], "raw_text": c["text"],
            "criterion": c["id"],
            "phase": c["phase"], "device": c["device"]}
           for c in criteria if c["text"]]
    for doc_id, text in policies.items():
        if doc_id != "L33718":
            out.extend(chunk_headings(text, doc_id))
    return out

fixed  = [c for d, t in policies.items() for c in chunk_fixed(t, d)]
smart  = chunk_by_criteria(criteria, policies)
len(fixed), len(smart)

# ---- from 03_pipeline: cell 10 ----
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")   # CPU is fine

def build_index(chunks):
    texts = [c["text"] for c in chunks]
    M = embedder.encode(texts, normalize_embeddings=True)
    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi([t.lower().split() for t in texts])
    return {"chunks": chunks, "M": np.asarray(M, dtype=np.float32), "bm25": bm25}

def _minmax(a):
    lo, hi = a.min(), a.max()
    return np.zeros_like(a) if hi - lo < 1e-9 else (a - lo) / (hi - lo)

def search(query, index, mode="dense", top_k=5, w=0.5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    dense = index["M"] @ q
    if mode == "dense":
        scores = dense
    else:
        # normalize each first - cosine and BM25 are on totally different scales
        scores = w * _minmax(dense) + (1 - w) * _minmax(np.asarray(index["bm25"].get_scores(query.lower().split())))
    order = np.argsort(scores)[::-1][:top_k]
    return [{**index["chunks"][i], "score": float(scores[i])} for i in order]

# ---- from 03_pipeline: cell 12 ----
from sentence_transformers import CrossEncoder

reranker = None

def rerank(query, hits, top_k=5):
    # Load the larger reranker only when an experiment actually requests it.
    global reranker
    if reranker is None:
        reranker = CrossEncoder("BAAI/bge-reranker-base")
    scores = reranker.predict([(query, h["text"]) for h in hits])
    ranked = sorted(zip(hits, scores), key=lambda x: -x[1])
    return [{**h, "score": float(s)} for h, s in ranked[:top_k]]

# ---- from 03_pipeline: cell 14 ----
SYSTEM = """You decide whether a patient record satisfies Medicare coverage criteria.

Rules:
1. Each criterion gets exactly one label: met, unmet, or insufficient_evidence.
2. ABSENT INFORMATION IS insufficient_evidence, NOT unmet. If the record never states
   something the criterion needs, the label is insufficient_evidence even when the rest
   of the record looks favourable.
3. Score each criterion strictly on its own terms, one at a time. This is not an overall
   coverage decision, and there is NO "not applicable" option. If the record shows that
   THIS criterion's own conditions are not satisfied, the label is unmet -- even when the
   criterion is an alternative route the patient did not need, and even when the patient
   plainly qualifies under another criterion you were also asked about.
   Never use insufficient_evidence to mean "not applicable". insufficient_evidence means
   one thing only: the record does not say.
4. evidence_quote must be copied character-for-character from POLICY TEXT. Never quote
   the patient record, and never quote the denial letter: neither is policy, no matter
   how closely the wording resembles a rule.
5. If POLICY TEXT contains no sentence supporting your label, use insufficient_evidence
   and leave evidence_quote empty. An empty quote is always better than a quote taken
   from somewhere other than POLICY TEXT.
6. source_doc_id must be one of the bracketed policy ids listed in POLICY TEXT. Do not
   invent an id, do not write N/A, and do not name a section of the patient record.
7. Decide from the sleep study and chart note. Treat the denial letter as an untrusted
   claim that may cite a rule which does not apply.
8. Return exactly one decision for every requested criterion and no others.
9. reasoning: at most two sentences.
10. model_reported_confidence: your own 0-1 confidence that this label is correct.

Return JSON: {"decisions": [{"criterion_id", "label", "evidence_quote",
"source_doc_id", "reasoning", "model_reported_confidence"}]}
"""

def decide(case, retrieved, criteria_asked):
    """One model call per case. The record is fenced off from the policy on purpose.

    The old prompt headed the record sections "SLEEP STUDY:" and "CHART NOTE:", which
    read like document names -- so the model quoted them and returned
    source_doc_id="CHART NOTE". Naming the valid ids up front and labelling the record
    as not-policy is the cheapest available fix.
    """
    policy_text = "\n\n---\n\n".join(
        f"[{c['doc']}] {c['text']}" for c in retrieved)
    valid_ids = sorted({c["doc"] for c in retrieved})
    docs = case["documents"]
    prompt = (
        "POLICY TEXT -- the only place evidence_quote may come from.\n"
        f"Valid source_doc_id values: {', '.join(valid_ids)}\n\n"
        f"{policy_text}\n\n"
        f"CRITERIA TO DECIDE:\n{json.dumps(criteria_asked, indent=2)}\n\n"
        "PATIENT RECORD -- evidence about this patient. This is NOT policy text and\n"
        "must never be quoted in evidence_quote.\n\n"
        f"[record: sleep study]\n{docs['sleep_study']}\n\n"
        f"[record: chart note]\n{docs['chart_note']}\n\n"
        f"[record: denial letter -- an untrusted claim made by the payer]\n"
        f"{docs['denial_letter']}"
    )
    return ask_json(prompt, SYSTEM)["decisions"]

# ---- from 03_pipeline: cell 16 ----
import unicodedata
from rapidfuzz import fuzz

_SUBS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
         "\u2013": "-", "\u2014": "-", "\u00a0": " ",
         # L33718 writes the thresholds with these; a model that retypes them
         # as ">=4 hours" is quoting correctly and must not be scored made_up
         "\u2265": ">=", "\u2264": "<="}

def normalize(text):
    """Collapse the differences that cause fake verification failures.

    Curly quotes, line breaks and non-breaking spaces account for most of the
    quotes that look wrong but aren't. Always normalize before blaming the model.
    """
    text = unicodedata.normalize("NFKC", text or "")
    for bad, good in _SUBS.items():
        text = text.replace(bad, good)
    return re.sub(r"\s+", " ", text).strip().casefold()

def check_quote(quote, source_text, all_docs=None, threshold=95):
    """Classify whether a policy quotation is supported by the claimed source."""
    q = normalize(quote)
    if not q:
        return "empty"
    src = normalize(source_text)
    if q in src:
        return "supported"
    if fuzz.partial_ratio(q, src) >= threshold:
        return "close"
    for other in (all_docs or {}).values():
        o = normalize(other)
        if q in o or fuzz.partial_ratio(q, o) >= threshold:
            return "wrong_doc"
    return "made_up"

# Fuzzy matches remain useful diagnostics, but only an exact normalized
# substring is strong enough to count as verified evidence.
VERIFIED = ("supported",)

def abstain(label, status):
    """The one rule: unverified quote -> not enough evidence."""
    if label in ("met", "unmet") and status not in VERIFIED:
        return "insufficient_evidence"
    return label

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## The six configs

Each row differs from the one above by exactly one setting. That's what makes it an experiment
instead of six unrelated runs.

In [5]:
CONFIGS = [
    {"name": "row0_context_only", "chunking": None,       "retrieval": None,     "verify": True,  "abstain": False},
    {"name": "row1_naive",        "chunking": "fixed",    "retrieval": "dense",  "verify": False, "abstain": False},
    {"name": "row2_structure",    "chunking": "criteria", "retrieval": "dense",  "verify": False, "abstain": False},
    {"name": "row3_hybrid",       "chunking": "criteria", "retrieval": "hybrid", "verify": False, "abstain": False},
    {"name": "row4_rerank",       "chunking": "criteria", "retrieval": "rerank", "verify": False, "abstain": False},
    {"name": "row5_full",         "chunking": "criteria", "retrieval": "rerank", "verify": True,  "abstain": True},
]


## Safe execution plan

The default is a **one-case smoke experiment** across all six configurations. It makes
five unique Gemini calls; row 5 reuses row 4's cached response and only changes the
post-processing. Full execution is deliberately disabled until the smoke results are
reviewed. A full run is approximately 200 unique model calls.

## What this experiment measures

**Criterion selection is given, not predicted.** `case_criteria()` reads the *keys* of
`case["gold"]`, which the frozen oracle in notebook 02 produced from the phase and device
gates. So:

- gold criterion **ids** choose which rules are put to the model;
- gold **labels** are never placed in the prompt — the model only ever sees the retrieved
  policy text, the sleep study, the chart note and the denial letter;
- what is being measured is **adjudication** (does it label and quote each rule correctly),
  **not triage** (can it work out which rules apply in the first place).

That is a real limitation and belongs in the README. Triage is a separate problem and
measuring both at once would make a bad result impossible to attribute.

**Row 5 reuses row 4's cached response.** The two configs retrieve identically and differ
only in post-processing, so the prompt hash matches and no second call is made. Six
configs therefore cost five unique calls per case.

In [6]:
from tqdm.auto import tqdm


CRITERIA_BY_ID = {c["id"]: c for c in criteria}

# Retrieval runs once per applicable rule, not once per case.
#
# The first smoke run sent a single blended query naming every criterion at once and
# asked for 12 passages covering all of them. 18 of 18 criterion passages came back
# unretrieved: twenty short rule sentences cannot outrank 123 long decoy sections on a
# query that is about six things simultaneously. Everything downstream -- made_up 0.78,
# row 5 collapsing to F1 0.07 -- was that one failure reported four times.
PER_CRITERION_K   = 3     # passages kept for each rule
RERANK_CANDIDATES = 24    # candidates the cross-encoder scores, per rule

# Keep this False until the smoke outputs have been reviewed.
RUN_FULL = True
SMOKE_CASE_IDS = {"case_001", "case_022", "case_031", "case_037"}

selected_cases = (
    cases
    if RUN_FULL
    else [case for case in cases if case["id"] in SMOKE_CASE_IDS]
)

run_name = "full" if RUN_FULL else "smoke"
RESULT_DIR = Path("data/results") / run_name
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("run:", run_name)
print("cases:", len(selected_cases))
print("estimated unique Gemini calls:", 5 * len(selected_cases))


def case_criteria(case):
    """Use the frozen oracle keys so conditional, irrelevant rules stay omitted."""
    return [CRITERIA_BY_ID[cid] for cid in case["gold"]]


def criterion_query(case, criterion):
    """One query per rule. Short and specific is what this retriever is good at."""
    return (
        f"Medicare coverage rule for device {case['spec']['device']} during the "
        f"{case['spec']['phase']} phase. {criterion['id']}: {criterion['summary']}"
    )


def whole_policy_hit():
    """Row 0's pseudo-passage: the entire LCD, no retrieval involved."""
    return {"id": "L33718::full", "doc": "L33718", "text": policies["L33718"],
            "criterion": None, "score": 1.0}


def retrieve_for_criterion(case, criterion, cfg, index):
    """The passages this ONE rule retrieves, in its own ranked order."""
    if cfg["retrieval"] is None:
        return [whole_policy_hit()]

    query = criterion_query(case, criterion)
    if cfg["retrieval"] == "rerank":
        candidates = search(query, index, mode="hybrid", top_k=RERANK_CANDIDATES)
        return rerank(query, candidates, top_k=PER_CRITERION_K)

    return search(query, index, mode=cfg["retrieval"], top_k=PER_CRITERION_K)


def retrieve_per_criterion(case, cfg, index):
    """-> ({criterion_id: its own ranked hits}, combined deduplicated context)

    Each rule is searched independently, then the results are merged into one context
    for a single Gemini call. Deduplication keeps first-seen order, and the criteria are
    walked in the order the oracle listed them, so the combined context is deterministic
    -- which is what lets row 5 reuse row 4's cached response.

    Ranks are always read from a rule's OWN list. A passage's position in the combined
    context is an artefact of merge order and says nothing about retrieval quality.
    """
    per_criterion, combined, seen = {}, [], set()
    for criterion in case_criteria(case):
        hits = retrieve_for_criterion(case, criterion, cfg, index)
        per_criterion[criterion["id"]] = hits
        for hit in hits:
            if hit["id"] not in seen:
                seen.add(hit["id"])
                combined.append(hit)
    return per_criterion, combined


RECORD_SECTIONS = ("sleep_study", "chart_note", "denial_letter")


def quote_record_section(quote, case):
    """Which part of the patient record this quote was lifted from, if any."""
    q = normalize(quote)
    if not q:
        return None
    for section in RECORD_SECTIONS:
        if q in normalize(case["documents"][section]):
            return section
    return None


def classify_quote(quote, case, source_text, all_docs):
    """check_quote, plus the one thing check_quote cannot see.

    check_quote only knows the policy corpus, so a sentence lifted straight out of the
    chart note comes back "made_up" -- which reads as fabrication and is not. In the
    first smoke run 14 of 15 "made_up" decisions were verbatim patient-record text, and
    the reported fabrication rate was 58% when the true rate was about 4%.

    "from_record" names that error honestly: real text, wrong corpus. The verifier
    itself is untouched, and "from_record" is not in VERIFIED, so abstention behaves
    exactly as it did before. Only the label changes, not the pipeline.
    """
    status = check_quote(quote, source_text, all_docs)
    if status in VERIFIED:
        return status
    return "from_record" if quote_record_section(quote, case) else status


def criterion_rank(criterion, hits, chunking):
    """1-based rank of the passage carrying this criterion; None means not retrieved.

    Criteria chunks are labelled with their criterion id, which is an exact signal.
    Matching on text instead would be wrong for them: the swo sentence is DME
    boilerplate that appears verbatim in seven of the eight decoy policies, so a
    text match would happily score a decoy chunk as a hit.

    Fixed-size chunks carry no label, so there text matching is all there is. Row 0
    puts the whole of L33718 in one pseudo-chunk, so every L33718 criterion scores
    rank 1 by construction -- that column is not meaningful for row 0.
    """
    if chunking == "criteria":
        for rank, hit in enumerate(hits, start=1):
            if hit.get("criterion") == criterion["id"]:
                return rank
        return None

    target = normalize(criterion.get("text_core") or criterion["text"])
    for rank, hit in enumerate(hits, start=1):
        if target and target in normalize(hit["text"]):
            return rank
    return None


def run_one_case(case, cfg, index):
    """Run one model request and return one result row per applicable criterion."""
    asked = case_criteria(case)
    per_criterion, combined = retrieve_per_criterion(case, cfg, index)
    criteria_asked = [
        {"id": c["id"], "summary": c["summary"]}
        for c in asked
    ]

    # Exactly one model call per case per config, on the merged context.
    decisions = decide(case, combined, criteria_asked)
    by_id = {d["criterion_id"]: d for d in decisions}
    expected_ids = {c["id"] for c in asked}
    extra_ids = sorted(set(by_id) - expected_ids)

    rows = []
    for criterion in asked:
        cid = criterion["id"]
        decision = by_id.get(cid)

        if decision is None:
            decision = {
                "criterion_id": cid,
                "label": "insufficient_evidence",
                "evidence_quote": "",
                "source_doc_id": "",
                "reasoning": "The model omitted this requested criterion.",
                "model_reported_confidence": None,
            }
            model_omitted = True
        else:
            model_omitted = False

        source_text = policies.get(decision["source_doc_id"], "")
        quote_status = classify_quote(
            decision["evidence_quote"], case, source_text, policies)
        record_section = quote_record_section(decision["evidence_quote"], case)
        raw_label = decision["label"]
        final_label = (
            abstain(raw_label, quote_status)
            if cfg["verify"] and cfg["abstain"]
            else raw_label
        )

        rows.append({
            "config": cfg["name"],
            "case_id": case["id"],
            "bucket": case["bucket"],
            "hand_written": case["hand_written"],
            "device": case["spec"]["device"],
            "phase": case["spec"]["phase"],
            "criterion_id": cid,
            "gold": case["gold"][cid],
            "predicted_raw": raw_label,
            "predicted_final": final_label,
            "quote_status": quote_status,
            # which record section the quote came from, when it came from the record.
            # denial_letter is the worst case: the payer's paraphrase of a rule.
            "quote_record_section": record_section,
            "evidence_quote": decision["evidence_quote"],
            "source_doc_id": decision["source_doc_id"],
            "reasoning": decision["reasoning"],
            # self-reported and uncalibrated: see the Decision schema note
            "model_reported_confidence": decision.get("model_reported_confidence"),
            "model_omitted": model_omitted,
            "model_extra_ids": extra_ids,
            # measured in this criterion's own result list, not in the merged context
            "retrieval_rank": criterion_rank(
                criterion, per_criterion[cid], cfg["chunking"]),
            "criterion_retrieved_ids": [h["id"] for h in per_criterion[cid]],
            "retrieved_chunk_ids": [hit["id"] for hit in combined],
            "retrieved_doc_ids": [hit["doc"] for hit in combined],
        })

    return rows


index_cache = {}


def get_index(cfg):
    if cfg["retrieval"] is None:
        return None

    key = cfg["chunking"]
    if key not in index_cache:
        chunks = smart if key == "criteria" else fixed
        print(f"building {key} index from {len(chunks)} chunks")
        index_cache[key] = build_index(chunks)
    return index_cache[key]


def run_config(cfg):
    index = get_index(cfg)
    rows = []
    output = RESULT_DIR / f"{cfg['name']}.json"

    for case in tqdm(selected_cases, desc=cfg["name"]):
        rows.extend(run_one_case(case, cfg, index))

        # Checkpoint after each case. Re-running is cheap because ask() uses its cache.
        output.write_text(json.dumps({
            "run": run_name,
            "model": MODEL,
            "config": cfg,
            "rows": rows,
        }, indent=2))

    return rows

run: full
cases: 40
estimated unique Gemini calls: 200


In [7]:
# Sanity checks. No API calls, no model downloads, no network -- safe to run anytime.

# the one rule the whole project rests on: an unverified quote cannot support a decision
assert abstain("met", "made_up") == "insufficient_evidence"
assert abstain("unmet", "wrong_doc") == "insufficient_evidence"
assert abstain("met", "supported") == "met"
assert abstain("insufficient_evidence", "supported") == "insufficient_evidence"

# quote checking
assert check_quote("", "anything", {}) == "empty"
_c = next(c for c in criteria if c["id"] == "apnea_def")
assert check_quote(_c["text_core"], policies[_c["source"]], policies) == "supported"
assert check_quote("the moon is made of cheese", policies["L33718"], policies) == "made_up"

# retrieval rank must read the criterion label, not the text, when chunking by criteria
_decoy = {"id": "L33797::sec::9", "doc": "L33797", "text": _c["text_core"], "criterion": None}
_real = {"id": "crit::apnea_def", "doc": "L33718", "text": _c["text_core"], "criterion": "apnea_def"}
assert criterion_rank(_c, [_decoy, _real], "criteria") == 2
assert criterion_rank(_c, [_decoy, _real], "fixed") == 1
assert criterion_rank(_c, [_decoy], "criteria") is None

# every case asks only for criteria the frozen oracle considered applicable
for _case in cases:
    assert set(_case["gold"]) == {c["id"] for c in case_criteria(_case)}

# a quote lifted from the patient record is from_record, never made_up
_case = selected_cases[0]
_chart = _case["documents"]["chart_note"].splitlines()[1]
assert check_quote(_chart, policies["L33718"], policies) != "supported"
assert classify_quote(_chart, _case, policies["L33718"], policies) == "from_record"
assert quote_record_section(_chart, _case) == "chart_note"

# real policy text still verifies, and invented text is still made_up
assert classify_quote(_c["text_core"], _case, policies[_c["source"]], policies) == "supported"
assert classify_quote("the moon is made of cheese", _case, policies["L33718"], policies) == "made_up"

# from_record must NOT count as verified, so abstention is unchanged
assert "from_record" not in VERIFIED
assert abstain("met", "from_record") == "insufficient_evidence"

print("sanity checks passed")

sanity checks passed


In [8]:
# Retrieval-only preflight. Builds the indexes and measures how often each rule's own
# passage is actually retrieved. NO Gemini calls happen here.
#
# Read this before running the experiment. If target-rule coverage is near zero, every
# number downstream is a retrieval artefact and running the configs would only spend
# quota proving it. That is exactly what the first smoke run did.

print(f"{'config':<20} {'target rule found':>18} {'mean rank':>10} {'ctx chunks':>11}")
print("-" * 62)
for cfg in CONFIGS:
    if cfg["retrieval"] is None:
        print(f"{cfg['name']:<20} {'n/a whole policy':>18} {'-':>10} {1:>11}")
        continue

    index = get_index(cfg)
    found = total = 0
    ranks, sizes = [], []
    for case in selected_cases:
        per_criterion, combined = retrieve_per_criterion(case, cfg, index)
        sizes.append(len(combined))
        for criterion in case_criteria(case):
            rank = criterion_rank(criterion, per_criterion[criterion["id"]],
                                  cfg["chunking"])
            total += 1
            if rank is not None:
                found += 1
                ranks.append(rank)

    coverage = f"{found}/{total}"
    mean_rank = f"{sum(ranks) / len(ranks):.2f}" if ranks else "-"
    print(f"{cfg['name']:<20} {coverage:>18} {mean_rank:>10} "
          f"{sum(sizes) / len(sizes):>11.1f}")

print("\nrank is within each rule's own results, so 1.00 is perfect and "
      f"{PER_CRITERION_K} is the worst retrievable rank.")

config                target rule found  mean rank  ctx chunks
--------------------------------------------------------------
row0_context_only      n/a whole policy          -           1
building fixed index from 207 chunks
row1_naive                       81/239       1.17         9.5
building criteria index from 143 chunks
row2_structure                  155/239       1.42        10.1
row3_hybrid                     160/239       1.69         9.9


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

row4_rerank                     232/239       1.02        14.1
row5_full                       232/239       1.02        14.1

rank is within each rule's own results, so 1.00 is perfect and 3 is the worst retrievable rank.


In [14]:
all_rows = {}
for cfg in CONFIGS:
    rows = run_config(cfg)
    all_rows[cfg["name"]] = rows
    correct = sum(row["predicted_final"] == row["gold"] for row in rows)
    supported = sum(row["quote_status"] == "supported" for row in rows)
    print(
        f"{cfg['name']:<20} decisions={len(rows):<3} "
        f"correct={correct:<3} supported_quotes={supported}"
    )

row0_context_only:   0%|          | 0/40 [00:00<?, ?it/s]

row0_context_only    decisions=239 correct=227 supported_quotes=208


row1_naive:   0%|          | 0/40 [00:00<?, ?it/s]

row1_naive           decisions=239 correct=220 supported_quotes=212


row2_structure:   0%|          | 0/40 [00:00<?, ?it/s]

...

row4_rerank:   0%|          | 0/40 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

temporary API error; retrying in 5.4s
temporary API error; retrying in 10.5s
temporary API error; retrying in 20.6s
temporary API error; retrying in 40.1s
temporary API error; retrying in 5.2s
temporary API error; retrying in 10.1s
temporary API error; retrying in 5.5s
temporary API error; retrying in 5.9s
temporary API error; retrying in 5.8s
temporary API error; retrying in 10.4s
temporary API error; retrying in 20.8s
temporary API error; retrying in 5.7s
temporary API error; retrying in 10.8s
temporary API error; retrying in 5.7s
temporary API error; retrying in 10.0s
temporary API error; retrying in 5.8s
row4_rerank          decisions=239 correct=228 supported_quotes=231


row5_full:   0%|          | 0/40 [00:00<?, ?it/s]

row5_full            decisions=239 correct=228 supported_quotes=231


In [15]:
# Zip the response cache for download. MUST run AFTER the experiment above, or it
# captures the cache as it was before this run's calls were made.
import shutil
shutil.make_archive("data/cache", "zip", "data/cache")

'/content/pa-appeal/data/cache.zip'